In [1]:
from tensorflow.keras.models import load_model
import pickle
from keras.preprocessing.sequence import pad_sequences
import numpy as np

# Load the tokenizer
with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

# Load the trained model
model = load_model('uci_sentimentanalysis.h5')

2025-12-06 02:54:12.674679: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-06 02:54:12.730156: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-06 02:54:15.384281: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-06 02:54:17.445902: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
def predict_sentiment(text):
    # The max length of the sequence must be the same as used during training
    # You can get this from X.shape[1] if you ran the training code above. 
    # Let's assume a max length based on the training data. If you have X, use X.shape[1].
    # For now, we'll try to infer it. A common value for this dataset is around 40-50.
    # If the tokenizer was saved, the max sequence length isn't stored, so we need to know it 
    # or re-calculate it from the original 'X' (X.shape[1]) if available.
    # Since X is available in the training notebook context:
    MAX_SEQUENCE_LENGTH = model.input_shape[1] 

    # 1. Tokenize the text
    tknz_text = tokenizer.texts_to_sequences([text])
    
    # 2. Pad the sequence
    padded_text = pad_sequences(tknz_text, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    
    # 3. Make prediction
    prediction = model.predict(padded_text)[0][0]
    
    # 4. Interpret the result
    # The sigmoid output is a probability between 0 and 1.
    # Typically, > 0.5 is positive (1), and <= 0.5 is negative (0).
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    
    # Return the prediction value and the interpreted sentiment
    return prediction, sentiment

In [3]:
# Example 1: A clearly positive review
test_sentence_1 = "This app is absolutely fantastic and works perfectly!"
score_1, result_1 = predict_sentiment(test_sentence_1)
print(f"Text: '{test_sentence_1}'")
print(f"Prediction Score (0=Negative, 1=Positive): {score_1:.4f}")
print(f"Result: {result_1}\n")

# Example 2: A clearly negative review
test_sentence_2 = "Worst experience, the service was terrible and I will not return."
score_2, result_2 = predict_sentiment(test_sentence_2)
print(f"Text: '{test_sentence_2}'")
print(f"Prediction Score (0=Negative, 1=Positive): {score_2:.4f}")
print(f"Result: {result_2}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step
Text: 'This app is absolutely fantastic and works perfectly!'
Prediction Score (0=Negative, 1=Positive): 0.4662
Result: Negative

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Text: 'Worst experience, the service was terrible and I will not return.'
Prediction Score (0=Negative, 1=Positive): 0.4662
Result: Negative
